# 1. Load the Cleaned Data

In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\processed"
FINAL     = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\final"

df = pd.read_csv(os.path.join(PROCESSED, "daedalus_merged.csv"))
print(f"Loaded: {df.shape}")
print(df.columns.tolist())

Loaded: (359, 17)
['country', 'city', 'aqi_value', 'aqi_category', 'pm2.5_aqi_value', 'meal_cheap_usd', 'meal_midrange_usd', 'rent_1br_outside_usd', 'rent_3br_city_usd', 'basic_utilities_usd', 'monthly_pass_usd', 'avg_net_salary_usd', 'popcity', 'densitycity', 'congestion2019', 'crime_index', 'safety_index']


# 2. Min-max normalizer (0 to 100, higher = better):


In [2]:
def normalize(series, invert=False):
    """
    Scales a series to 0-100.
    invert=True when lower raw value = better score (e.g. crime, AQI, rent)
    """
    mn, mx = series.min(), series.max()
    scaled = (series - mn) / (mx - mn) * 100
    if invert:
        scaled = 100 - scaled
    return scaled.round(2)

# 3. Pillar 1: Air Quality Score

In [3]:
# Lower AQI = better air = higher score → invert both
df['aqi_score']    = normalize(df['aqi_value'],       invert=True)
df['pm25_score']   = normalize(df['pm2.5_aqi_value'], invert=True)

# Weighted average within pillar (PM2.5 is more health-relevant)
df['air_quality_score'] = (df['aqi_score'] * 0.4 + df['pm25_score'] * 0.6).round(2)

print("Air quality score sample:")
print(df[['city', 'aqi_value', 'pm2.5_aqi_value', 'air_quality_score']]\
      .sort_values('air_quality_score', ascending=False).head(10))

Air quality score sample:
           city  aqi_value  pm2.5_aqi_value  air_quality_score
153    brasilia         21               11              99.02
76     canberra         24               11              98.77
351     uppsala         29                9              98.62
31    stockholm         31                8              98.60
354       turku         36                5              98.59
3      tauranga         19               17              98.37
167        cork         27               13              98.25
54      tampere         30               12              98.13
117    limerick         23               17              98.04
159  wellington         24               17              97.95


# 4. Pillar 2a: Absolute cost score (lower cost = better):

In [4]:
# Create a composite cost index first
# Weighted sum of key cost indicators
df['cost_index_raw'] = (
    df['meal_cheap_usd']       * 0.20 +
    df['rent_1br_outside_usd'] * 0.50 +
    df['basic_utilities_usd']  * 0.20 +
    df['monthly_pass_usd']     * 0.10
)

# Lower cost = better score → invert
df['cost_score_absolute'] = normalize(df['cost_index_raw'], invert=True).round(2)

print("Absolute cost score (lower cost cities rank higher):")
print(df[['city', 'country', 'cost_index_raw', 'cost_score_absolute']]\
      .sort_values('cost_score_absolute', ascending=False).head(10))

Absolute cost score (lower cost cities rank higher):
          city    country  cost_index_raw  cost_score_absolute
103      medan  Indonesia          44.462               100.00
75     kolkata      India          49.600                99.79
87       adana     Turkey          57.945                99.46
203    colombo  Sri Lanka          59.961                99.38
101      konya     Turkey          63.111                99.25
357      cairo      Egypt          66.168                99.13
40       brest     France          72.284                98.88
143  gaziantep     Turkey          72.877                98.86
144       pune      India          74.298                98.80
183  bengaluru      India          80.397                98.56


# 5.  Pillar 2b: Affordability ratio score (cost vs local salary):

In [5]:
# Affordability = what % of monthly salary does basic living cost?
# Lower % = more affordable relative to local wages
df['monthly_living_cost'] = (
    df['meal_cheap_usd']       * 30  +   # ~30 cheap meals/month
    df['rent_1br_outside_usd']       +   # monthly rent
    df['basic_utilities_usd']        +   # utilities
    df['monthly_pass_usd']               # transport
)

df['affordability_ratio'] = (df['monthly_living_cost'] / df['avg_net_salary_usd'] * 100).round(2)

# Lower ratio = more affordable → invert
df['cost_score_affordability'] = normalize(df['affordability_ratio'], invert=True).round(2)

print("Affordability ratio score (cost relative to local salary):")
print(df[['city', 'country', 'monthly_living_cost', 'avg_net_salary_usd',
          'affordability_ratio', 'cost_score_affordability']]\
      .sort_values('cost_score_affordability', ascending=False).head(10))

Affordability ratio score (cost relative to local salary):
            city                                            country  \
95       dunedin                           United States of America   
130        mecca                                       Saudi Arabia   
168        tulsa                           United States of America   
183    bengaluru                                              India   
296      el paso                           United States of America   
243    milwaukee                           United States of America   
170      qingdao                                              China   
64   albuquerque                           United States of America   
75       kolkata                                              India   
119      preston  United Kingdom of Great Britain and Northern I...   

     monthly_living_cost  avg_net_salary_usd  affordability_ratio  \
95               1070.63             4927.66                21.73   
130               682

# 6. Pillar 3: Safety score:

In [6]:
# Safety index is already 0-100 (higher = safer), just normalize range
# Crime index is inverse (higher = more crime)
df['safety_score'] = (
    normalize(df['safety_index'], invert=False) * 0.6 +
    normalize(df['crime_index'],  invert=True)  * 0.4
).round(2)

print("Safety score sample:")
print(df[['city', 'country', 'crime_index', 'safety_index', 'safety_score']]\
      .sort_values('safety_score', ascending=False).head(10))

Safety score sample:
          city               country  crime_index  safety_index  safety_score
59   abu dhabi  United Arab Emirates         11.2          88.8        100.00
340       doha                 Qatar         14.5          85.5         95.44
293     taipei                 China         15.1          84.9         94.61
292     taipei                 China         15.1          84.9         94.61
221    sharjah  United Arab Emirates         15.9          84.1         93.51
24       dubai  United Arab Emirates         16.4          83.6         92.82
42        bern           Switzerland         17.7          82.3         91.02
256     zurich           Switzerland         18.3          81.7         90.19
115     munich               Germany         18.8          81.2         89.50
51   trondheim                Norway         19.1          80.9         89.09


# 7. Pillar 4: Urban density score:

In [7]:
# Density is nuanced — very high AND very low density are both bad
# Sweet spot is moderate density (walkable but not overcrowded)
# We use a "moderate density preference" curve instead of simple invert

density_median = df['densitycity'].median()
density_std    = df['densitycity'].std()

# Score peaks at median density, falls off on both sides
df['density_score_raw'] = np.exp(
    -0.5 * ((df['densitycity'] - density_median) / density_std) ** 2
) * 100

# Congestion — lower is better
df['congestion_score'] = normalize(df['congestion2019'], invert=True)

# Combined urban score
df['urban_score'] = (
    df['density_score_raw'] * 0.5 +
    df['congestion_score']  * 0.5
).round(2)

print("Urban score sample:")
print(df[['city', 'densitycity', 'congestion2019', 'urban_score']]\
      .sort_values('urban_score', ascending=False).head(10))

Urban score sample:
            city  densitycity  congestion2019  urban_score
333     syracuse  2236.719840             8.0        98.92
145   greensboro   776.646573             7.0        98.40
269    cleveland  1970.928904             9.0        97.85
243    milwaukee  2387.774876             9.0        97.81
136    worcester  1870.957205             9.0        97.81
69   minneapolis  2735.464988             9.0        97.54
8       richmond  1315.768898             9.0        97.29
7       richmond  1315.768898             9.0        97.29
271  little rock   622.429782             8.0        96.96
322    rochester  2273.068299            10.0        96.79


# 8. Final composite livability score:

In [14]:
import re

# ── country lookup fix ─────────────────────────────────────
crime_lookup = pd.read_csv(r"C:\Users\alokhande\Desktop\Project_Daedalus\data\raw\crime-index-2023.csv")
crime_lookup.columns = crime_lookup.columns.str.lower()
crime_lookup['city'] = crime_lookup['city'].str.strip().str.lower()
crime_lookup = crime_lookup[['city','country']].drop_duplicates(subset=['city'])

aqi_lookup = pd.read_csv(r"C:\Users\alokhande\Desktop\Project_Daedalus\data\raw\global air pollution dataset.csv")
aqi_lookup.columns = aqi_lookup.columns.str.lower()
aqi_lookup['city'] = aqi_lookup['city'].str.strip().str.lower()
aqi_lookup = aqi_lookup[['city','country']].drop_duplicates(subset=['city'])

full_lookup = pd.concat([crime_lookup, aqi_lookup]).drop_duplicates(subset=['city'])

manual_countries = {
    'taipei':    'Taiwan',
    'bergen':    'Norway',
    'stavanger': 'Norway',
    'baku':      'Azerbaijan',
}

def fix_country(c):
    if pd.isna(c): return 'Unknown'
    if re.match(r'^[A-Z]{2}$', str(c).strip()): return 'United States of America'
    return c

df = df.drop(columns=['country'], errors='ignore')
df = df.merge(full_lookup, on='city', how='left')
df['country'] = df['country'].apply(fix_country)
df['country'] = df['city'].map(manual_countries).fillna(df['country'])

# ── recompute scores ───────────────────────────────────────
WEIGHTS = {
    'air_quality': 0.25,
    'cost':        0.30,
    'safety':      0.25,
    'urban':       0.20,
}

df['livability_score_absolute'] = (
    df['air_quality_score']    * WEIGHTS['air_quality'] +
    df['cost_score_absolute']  * WEIGHTS['cost']        +
    df['safety_score']         * WEIGHTS['safety']      +
    df['urban_score']          * WEIGHTS['urban']
).round(2)

df['livability_score_affordability'] = (
    df['air_quality_score']         * WEIGHTS['air_quality'] +
    df['cost_score_affordability']  * WEIGHTS['cost']        +
    df['safety_score']              * WEIGHTS['safety']      +
    df['urban_score']               * WEIGHTS['urban']
).round(2)

df['rank_absolute']      = df['livability_score_absolute'].rank(ascending=False).astype(int)
df['rank_affordability'] = df['livability_score_affordability'].rank(ascending=False).astype(int)

# ── verify country fixes ───────────────────────────────────
print("Country fix verification:")
print(df[df['city'].isin(['stavanger','bergen','taipei','antalya','bursa','baku'])]\
      [['city','country','rent_1br_outside_usd','basic_utilities_usd','avg_net_salary_usd']])

# ── print rankings ─────────────────────────────────────────
print("\n" + "="*60)
print("TOP 20 — Absolute cost model")
print("="*60)
print(df[['rank_absolute','city','country','livability_score_absolute']]\
      .sort_values('rank_absolute').head(20).to_string(index=False))

print("\n" + "="*60)
print("TOP 20 — Affordability model")
print("="*60)
print(df[['rank_affordability','city','country','livability_score_affordability']]\
      .sort_values('rank_affordability').head(20).to_string(index=False))

# ── save ──────────────────────────────────────────────────
FINAL = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\final"
df.to_csv(os.path.join(FINAL, "daedalus_scored.csv"), index=False)
print(f"\nSaved → daedalus_scored.csv")
print(f"Shape: {df.shape}")

Country fix verification:
          city     country  rent_1br_outside_usd  basic_utilities_usd  \
29     antalya      Turkey                347.28                72.49   
74       bursa      Turkey                175.27               106.61   
84      bergen      Norway                857.97               268.94   
182       baku  Azerbaijan                181.70                50.18   
278  stavanger      Norway                802.74               219.50   
292     taipei      Taiwan                441.34                82.34   
293     taipei      Taiwan                441.34                82.34   

     avg_net_salary_usd  
29               327.95  
74               331.46  
84              3218.31  
182              405.72  
278             3336.77  
292             1876.87  
293             1876.87  

TOP 20 — Absolute cost model
 rank_absolute      city         country  livability_score_absolute
             1   tampere         Finland                      90.26
             2 